<a href="https://colab.research.google.com/github/Phoenix23111/Universal-Monocular-Metric-Depth-Estimation/blob/master/LMAH_Depth_Estimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
=============================================================================
  Proof of Concept: Universal Monocular Metric Depth Estimation with LMAH
=============================================================================
"""
!pip install -q transformers datasets torch torchvision matplotlib pillow requests

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import itertools
import requests
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from torch.utils.data import DataLoader, Dataset
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from PIL import Image

# ──────────────────────────────────────────────
# 0.  Config
# ──────────────────────────────────────────────
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
NUM_SAMPLES = 50
NUM_EPOCHS  = 5
BATCH_SIZE  = 4
LR          = 1e-3
INPUT_SIZE  = (224, 224)
DEPTH_MAX_M = 10.0
MODEL_CKPT  = "nielsr/depth-anything-small"
OUTPUT_IMG  = "qualitative_results.png"

print(f"[INFO] Device : {DEVICE}")
print(f"[INFO] PyTorch: {torch.__version__}")


# ──────────────────────────────────────────────
# 1.  Dataset — load via Parquet API (no script)
# ──────────────────────────────────────────────
def _get_parquet_urls(dataset_id: str, split: str = "train") -> list:
    """
    Query the HF datasets-server to get the auto-generated Parquet URLs
    for a given dataset + split. This works even when the dataset has a
    legacy loading script, because HF converts every public dataset to
    Parquet automatically on refs/convert/parquet.
    """
    api = f"https://datasets-server.huggingface.co/parquet?dataset={dataset_id}"
    resp = requests.get(api, timeout=15)
    resp.raise_for_status()
    files = resp.json().get("parquet_files", [])
    urls  = [f["url"] for f in files if f["split"] == split]
    if not urls:
        raise ValueError(f"No Parquet files found for split='{split}'")
    return urls


def load_nyu_samples(n: int = NUM_SAMPLES) -> list:
    """
    Returns a plain Python list of dicts:
        [{"image": PIL.Image, "depth_map": PIL.Image}, ...]

    Strategy (in order):
      1. Fetch Parquet URLs via HF datasets-server API for sayakpaul/nyu_depth_v2
         → load with load_dataset("parquet", streaming=True)   [NO script needed]
      2. Synthetic fallback if HF is unreachable
    """
    from datasets import load_dataset

    DATASET_ID    = "sayakpaul/nyu_depth_v2"
    DEPTH_COL     = "depth_map"    # column name in this dataset
    IMAGE_COL     = "image"

    try:
        print(f"[DATA] Fetching Parquet URLs for {DATASET_ID} …")
        urls = _get_parquet_urls(DATASET_ID, split="train")
        print(f"[DATA] Found {len(urls)} Parquet shard(s) — using first shard only")

        # Load only the FIRST shard — it already has ~47k rows, far more than 50
        # streaming=True means only the row-groups we iterate over are downloaded
        ds = load_dataset(
            "parquet",
            data_files={"train": urls[0]},   # ← first shard, not all shards
            split="train",
            streaming=True,
        )

        samples = []
        for item in itertools.islice(ds, n):
            img = item[IMAGE_COL]
            dep = item[DEPTH_COL]
            # Normalise to PIL if needed
            if not isinstance(img, Image.Image):
                img = Image.fromarray(np.array(img))
            if not isinstance(dep, Image.Image):
                dep = Image.fromarray(np.array(dep))
            samples.append({"image": img, "depth_map": dep})

        print(f"[DATA] ✓ Streamed {len(samples)} samples (no full download)")
        return samples

    except Exception as e:
        print(f"[DATA] ✗ HF source failed: {e}")
        print("[DATA] Falling back to synthetic data …")
        return _synthetic_samples(n)


def _synthetic_samples(n: int, h: int = 480, w: int = 640) -> list:
    """Procedurally generated RGB + depth pairs for offline testing."""
    rng = np.random.default_rng(42)
    out = []
    for _ in range(n):
        rgb   = Image.fromarray(rng.integers(30, 220, (h, w, 3), dtype=np.uint8))
        x     = np.linspace(0.5, 8.0, w, dtype=np.float32)
        depth = np.clip(
            np.tile(x, (h, 1)) + rng.random((h, w)).astype(np.float32) * 1.5,
            0.1, DEPTH_MAX_M
        )
        # Store as float32 PIL — no deprecated 'mode' kwarg
        depth_pil = Image.fromarray(depth)          # mode auto-detected as 'F'
        out.append({"image": rgb, "depth_map": depth_pil})
    print(f"[DATA] Synthetic: {n} samples ready (640×480, depth 0.5–8 m)")
    return out


# ──────────────────────────────────────────────
# 2.  PyTorch Dataset
# ──────────────────────────────────────────────
class NYUMicroBenchmark(Dataset):
    """
    Wraps the pre-fetched list of samples.
    Returns (pixel_values, depth_gt) per item.
    """
    def __init__(self, samples: list, processor):
        self.samples   = samples
        self.processor = processor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]

        # ── RGB ───────────────────────────────────────────────────────────
        image = item["image"]
        if not isinstance(image, Image.Image):
            image = Image.fromarray(np.array(image))
        pv = self.processor(
            images=image.convert("RGB"), return_tensors="pt"
        )["pixel_values"].squeeze(0)                           # (3, H, W)

        # ── Depth → metres ────────────────────────────────────────────────
        dep = item["depth_map"]
        if not isinstance(dep, Image.Image):
            dep = Image.fromarray(np.array(dep))
        dep_np = np.array(dep, dtype=np.float32)

        # Auto-detect mm vs m by median of valid pixels
        valid  = dep_np[dep_np > 0]
        median = float(np.median(valid)) if len(valid) else 1.0
        if median > 20.0:
            dep_np /= 1000.0             # mm → metres

        dep_np = np.clip(dep_np, 0.1, DEPTH_MAX_M)

        # Resize to INPUT_SIZE
        dep_resized = np.array(
            Image.fromarray(dep_np).resize(INPUT_SIZE, Image.BILINEAR),
            dtype=np.float32
        )
        dep_t = torch.tensor(dep_resized).unsqueeze(0)         # (1, H, W)

        return pv, dep_t


# ──────────────────────────────────────────────
# 3.  Frozen backbone
# ──────────────────────────────────────────────
def get_frozen_backbone(ckpt: str):
    processor = AutoImageProcessor.from_pretrained(ckpt)
    model     = AutoModelForDepthEstimation.from_pretrained(ckpt)
    for p in model.parameters():
        p.requires_grad = False
    model.eval().to(DEVICE)
    n = sum(p.numel() for p in model.parameters())
    print(f"[INFO] Backbone loaded & frozen — {n:,} params")
    return model, processor


@torch.no_grad()
def get_relative_depth(backbone, pixel_values: torch.Tensor) -> torch.Tensor:
    """Forward pass → per-sample normalised depth (B, 1, H, W) ∈ [0, 1]."""
    rel = backbone(pixel_values=pixel_values).predicted_depth  # (B, H', W')
    B   = rel.shape[0]
    f   = rel.view(B, -1)
    mn  = f.min(1).values.view(B, 1, 1)
    mx  = f.max(1).values.view(B, 1, 1)
    rel = (rel - mn) / (mx - mn + 1e-8)
    return F.interpolate(
        rel.unsqueeze(1), size=INPUT_SIZE,
        mode="bilinear", align_corners=False
    )                                                          # (B, 1, H, W)


# ──────────────────────────────────────────────
# 4.  LMAH — Lightweight Metric Adaptation Head
# ──────────────────────────────────────────────
class LMAH(nn.Module):
    """
    ~2,400 trainable params.
    Conv(1→16, 3×3) → GELU → Conv(16→16, 3×3) → GELU → Conv(16→1, 1×1) → Softplus
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,  16, 3, padding=1), nn.GELU(),
            nn.Conv2d(16, 16, 3, padding=1), nn.GELU(),
            nn.Conv2d(16,  1, 1),
        )
        self.act = nn.Softplus()
        n = sum(p.numel() for p in self.parameters())
        print(f"[INFO] LMAH — {n:,} trainable params")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.net(x))


# ──────────────────────────────────────────────
# 5.  Training  (5 epochs, MSE)
# ──────────────────────────────────────────────
def train_lmah(lmah, backbone, loader):
    lmah.train()
    opt    = torch.optim.Adam(lmah.parameters(), lr=LR)
    loss_fn = nn.MSELoss()

    print(f"\n{'─'*50}")
    print(f"  TRAINING  epochs={NUM_EPOCHS}  batch={BATCH_SIZE}  lr={LR}")
    print(f"{'─'*50}")
    for ep in range(1, NUM_EPOCHS + 1):
        tot, n = 0.0, 0
        for pv, gt in loader:
            pv, gt = pv.to(DEVICE), gt.to(DEVICE)
            with torch.no_grad():
                rel = get_relative_depth(backbone, pv)
            loss = loss_fn(lmah(rel), gt)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); n += 1
        print(f"  Epoch [{ep}/{NUM_EPOCHS}]  MSE = {tot/n:.4f} m²")
    print(f"{'─'*50}\n")
    return lmah


# ──────────────────────────────────────────────
# 6.  Metrics
# ──────────────────────────────────────────────
def median_align(pred: np.ndarray, gt: np.ndarray) -> np.ndarray:
    return pred * (np.median(gt) / (np.median(pred) + 1e-8))

def compute_metrics(pred: np.ndarray, gt: np.ndarray, thr: float = 1.25):
    m = gt > 0.1
    p, g  = pred[m], gt[m]
    rmse  = float(np.sqrt(np.mean((p - g) ** 2)))
    ratio = np.maximum(p / (g + 1e-8), g / (p + 1e-8))
    delta = float((ratio < thr).mean() * 100)
    return rmse, delta


# ──────────────────────────────────────────────
# 7.  4-panel qualitative figure
# ──────────────────────────────────────────────
def save_figure(rgb, gt, base, lmah_out, b_rmse, b_d, l_rmse, l_d):
    vmin, vmax = gt.min(), gt.max()
    fig = plt.figure(figsize=(22, 5))
    fig.suptitle(
        "Qualitative Depth Estimation Results — PoC\n"
        "Frozen DepthAnything-Small  +  LMAH Metric Adaptation Head",
        fontsize=13, fontweight="bold", y=1.03,
    )
    gs = gridspec.GridSpec(1, 4, figure=fig, wspace=0.06)

    def panel(pos, data, title, is_rgb=False, highlight=False):
        ax = fig.add_subplot(gs[pos])
        if is_rgb:
            ax.imshow(data)
        else:
            im = ax.imshow(data, cmap="plasma", vmin=vmin, vmax=vmax)
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="metres")
        ax.set_title(title, fontsize=11,
                     color="#1a7fbb" if highlight else "black",
                     fontweight="bold" if is_rgb else "normal")
        ax.axis("off")

    panel(0, rgb,       "Input RGB", is_rgb=True)
    panel(1, gt,        "Ground Truth Depth (m)")
    panel(2, base,      f"Baseline (DepthAnything scaled)\n"
                        f"RMSE={b_rmse:.3f} m   δ<1.25={b_d:.1f}%")
    panel(3, lmah_out,  f"Ours — LMAH Head (5 ep.)\n"
                        f"RMSE={l_rmse:.3f} m   δ<1.25={l_d:.1f}%",
          highlight=True)

    plt.tight_layout()
    plt.savefig(OUTPUT_IMG, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[INFO] Figure saved → {OUTPUT_IMG}")


# ══════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════
def main():

    # 1. Stream 50 samples via Parquet API (no loading script)
    print("\n[STEP 1] Loading dataset …")
    samples = load_nyu_samples(NUM_SAMPLES)

    # 2. Frozen backbone
    print("\n[STEP 2] Loading frozen backbone …")
    backbone, processor = get_frozen_backbone(MODEL_CKPT)

    # 3. DataLoader
    print("\n[STEP 3] Building DataLoader …")
    ds     = NYUMicroBenchmark(samples, processor)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=0, pin_memory=(DEVICE == "cuda"))
    print(f"         {len(ds)} samples | {len(loader)} batches/epoch")

    # 4. Train LMAH
    print("\n[STEP 4] Training LMAH …")
    lmah = train_lmah(LMAH().to(DEVICE), backbone, loader)

    # 5. Evaluate on held-out test sample (last one)
    print("[STEP 5] Evaluating …\n")
    pv_t, gt_t = ds[NUM_SAMPLES - 1]
    pv = pv_t.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        rel = get_relative_depth(backbone, pv)
        lmah.eval()
        pred = lmah(rel)

    gt_np   = gt_t.squeeze().numpy()
    rel_np  = rel.squeeze().cpu().numpy()
    lmah_np = pred.squeeze().cpu().numpy()
    base_np = median_align(rel_np, gt_np)

    b_rmse, b_d = compute_metrics(base_np, gt_np)
    l_rmse, l_d = compute_metrics(lmah_np, gt_np)

    W = 58
    print("═" * W)
    print("  EVALUATION RESULTS  (1 held-out test image, NYU Depth V2)")
    print("═" * W)
    print(f"  {'Model':<34} {'RMSE ↓':>9}  {'δ<1.25 ↑':>10}")
    print("─" * W)
    print(f"  {'Baseline — DepthAnything (scaled)':<34} {b_rmse:>8.4f}m  {b_d:>9.2f}%")
    print(f"  {'Ours — LMAH Head  (5 epochs)':<34} {l_rmse:>8.4f}m  {l_d:>9.2f}%")
    print("═" * W)
    Δr, Δd = b_rmse - l_rmse, l_d - b_d
    print(f"\n  LMAH vs Baseline  RMSE  : {'↓' if Δr > 0 else '↑'} {abs(Δr):.4f} m")
    print(f"  LMAH vs Baseline  δ<1.25: {'↑' if Δd > 0 else '↓'} {abs(Δd):.2f}%\n")

    # 6. Save figure
    print("[STEP 6] Saving figure …")
    mean = np.array([0.485, 0.456, 0.406], np.float32)
    std  = np.array([0.229, 0.224, 0.225], np.float32)
    rgb  = np.clip(pv_t.numpy().transpose(1, 2, 0) * std + mean, 0, 1)
    save_figure(rgb, gt_np, base_np, lmah_np, b_rmse, b_d, l_rmse, l_d)

    print("\n[DONE] Complete. Check qualitative_results.png")


if __name__ == "__main__":
    main()